# SOPR Entry Point Deep Dive

**Goal:** Understand what happens AFTER we enter. This will help us:
1. Validate that entries are actually good
2. Design optimal exit strategies based on real data
3. Identify which entries we should skip

## Questions to Answer
- How far does price typically drop after entry? (max drawdown)
- How far does price typically rise after entry? (max gain)
- How long until max gain? (optimal hold period)
- Are there patterns in good vs bad entries?

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("Ready!")

In [ ]:
# Load data
DATA_DIR = Path("../data/raw")

sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")

df = sopr.join(sopr_sth, how='inner').join(price, how='inner').sort_index()
df = df[df.index >= '2018-12-15']  # 2019+

close = df['price']
print(f"Data: {len(df)} rows, {df.index.min().date()} to {df.index.max().date()}")

In [ ]:
# Entry signal
both_below_1 = (df['sopr'] < 1) & (df['sopr_sth'] < 1)
entries = both_below_1 & ~both_below_1.shift(1).fillna(False)

entry_dates = entries[entries].index.tolist()
print(f"Total entry signals: {len(entry_dates)}")

---
## 1. Analyze What Happens After Each Entry

In [ ]:
def analyze_entry(entry_date, close, lookahead_days=180):
    """
    Analyze what happens after an entry.
    Returns stats about the price path following entry.
    """
    entry_idx = close.index.get_loc(entry_date)
    entry_price = close.iloc[entry_idx]
    
    # Get price path for lookahead period
    end_idx = min(entry_idx + lookahead_days, len(close) - 1)
    future_prices = close.iloc[entry_idx:end_idx + 1]
    
    if len(future_prices) < 2:
        return None
    
    # Calculate returns from entry
    returns_from_entry = (future_prices - entry_price) / entry_price
    
    # Max drawdown (worst point after entry)
    max_drawdown = returns_from_entry.min()
    max_drawdown_idx = returns_from_entry.idxmin()
    days_to_max_dd = (max_drawdown_idx - entry_date).days
    
    # Max gain (best point after entry)
    max_gain = returns_from_entry.max()
    max_gain_idx = returns_from_entry.idxmax()
    days_to_max_gain = (max_gain_idx - entry_date).days
    
    # Returns at specific intervals
    def get_return_at_day(n):
        if entry_idx + n < len(close):
            return (close.iloc[entry_idx + n] - entry_price) / entry_price
        return np.nan
    
    return {
        'entry_date': entry_date,
        'entry_price': entry_price,
        'max_drawdown': max_drawdown,
        'days_to_max_dd': days_to_max_dd,
        'max_gain': max_gain,
        'days_to_max_gain': days_to_max_gain,
        'return_7d': get_return_at_day(7),
        'return_14d': get_return_at_day(14),
        'return_30d': get_return_at_day(30),
        'return_60d': get_return_at_day(60),
        'return_90d': get_return_at_day(90),
        'return_180d': get_return_at_day(180),
        'price_path': future_prices,
        'returns_path': returns_from_entry
    }

In [ ]:
# Analyze all entries
entry_analysis = []

for entry_date in entry_dates:
    result = analyze_entry(entry_date, close, lookahead_days=180)
    if result:
        entry_analysis.append(result)

print(f"Analyzed {len(entry_analysis)} entries")

In [ ]:
# Create summary dataframe
summary_df = pd.DataFrame([{
    'entry_date': e['entry_date'],
    'entry_price': e['entry_price'],
    'max_drawdown': e['max_drawdown'],
    'days_to_max_dd': e['days_to_max_dd'],
    'max_gain': e['max_gain'],
    'days_to_max_gain': e['days_to_max_gain'],
    'return_7d': e['return_7d'],
    'return_14d': e['return_14d'],
    'return_30d': e['return_30d'],
    'return_60d': e['return_60d'],
    'return_90d': e['return_90d'],
    'return_180d': e['return_180d']
} for e in entry_analysis])

print("\nENTRY ANALYSIS SUMMARY")
print("="*80)
print(summary_df.describe().round(3).to_string())

---
## 2. Visualize All Entry Outcomes

In [ ]:
# Plot all price paths from entry (normalized to entry = 0%)
fig = go.Figure()

for i, entry in enumerate(entry_analysis):
    returns = entry['returns_path']
    days = range(len(returns))
    
    # Color by outcome
    final_return = entry['return_90d'] if not np.isnan(entry['return_90d']) else returns.iloc[-1]
    color = 'green' if final_return > 0 else 'red'
    opacity = 0.3
    
    fig.add_trace(go.Scatter(
        x=list(days),
        y=returns.values * 100,
        mode='lines',
        line=dict(color=color, width=1),
        opacity=opacity,
        showlegend=False,
        hovertemplate=f"{entry['entry_date'].strftime('%Y-%m-%d')}<br>" +
                      "Day %{x}<br>Return: %{y:.1f}%<extra></extra>"
    ))

# Add zero line
fig.add_hline(y=0, line_dash='dash', line_color='black')

# Add median path
max_days = max(len(e['returns_path']) for e in entry_analysis)
median_returns = []
for d in range(max_days):
    day_returns = [e['returns_path'].iloc[d] if d < len(e['returns_path']) else np.nan 
                   for e in entry_analysis]
    median_returns.append(np.nanmedian(day_returns))

fig.add_trace(go.Scatter(
    x=list(range(len(median_returns))),
    y=[r * 100 for r in median_returns],
    mode='lines',
    line=dict(color='blue', width=3),
    name='Median Path'
))

fig.update_layout(
    title='All Entry Outcomes: Return from Entry Over Time<br><sup>Green=Profitable at 90d, Red=Loss at 90d, Blue=Median</sup>',
    xaxis_title='Days After Entry',
    yaxis_title='Return from Entry (%)',
    height=600
)
fig.show()

In [ ]:
# Distribution of max drawdown and max gain
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=['Max Drawdown Distribution', 'Max Gain Distribution',
                                   'Days to Max Drawdown', 'Days to Max Gain'])

fig.add_trace(go.Histogram(x=summary_df['max_drawdown']*100, nbinsx=30,
                           marker_color='red', name='Max DD'), row=1, col=1)
fig.add_trace(go.Histogram(x=summary_df['max_gain']*100, nbinsx=30,
                           marker_color='green', name='Max Gain'), row=1, col=2)
fig.add_trace(go.Histogram(x=summary_df['days_to_max_dd'], nbinsx=30,
                           marker_color='orange', name='Days to DD'), row=2, col=1)
fig.add_trace(go.Histogram(x=summary_df['days_to_max_gain'], nbinsx=30,
                           marker_color='blue', name='Days to Gain'), row=2, col=2)

fig.update_layout(height=600, showlegend=False, title_text='Entry Outcome Distributions')
fig.show()

print("\nKEY STATISTICS")
print("="*50)
print(f"\nMax Drawdown:")
print(f"  Median: {summary_df['max_drawdown'].median()*100:.1f}%")
print(f"  Mean: {summary_df['max_drawdown'].mean()*100:.1f}%")
print(f"  Worst: {summary_df['max_drawdown'].min()*100:.1f}%")
print(f"  75th percentile: {summary_df['max_drawdown'].quantile(0.75)*100:.1f}%")

print(f"\nMax Gain:")
print(f"  Median: {summary_df['max_gain'].median()*100:.1f}%")
print(f"  Mean: {summary_df['max_gain'].mean()*100:.1f}%")
print(f"  Best: {summary_df['max_gain'].max()*100:.1f}%")
print(f"  25th percentile: {summary_df['max_gain'].quantile(0.25)*100:.1f}%")

print(f"\nTiming:")
print(f"  Median days to max drawdown: {summary_df['days_to_max_dd'].median():.0f}")
print(f"  Median days to max gain: {summary_df['days_to_max_gain'].median():.0f}")

---
## 3. Returns at Different Time Horizons

In [ ]:
# Box plot of returns at different holding periods
periods = ['return_7d', 'return_14d', 'return_30d', 'return_60d', 'return_90d', 'return_180d']
period_labels = ['7 Days', '14 Days', '30 Days', '60 Days', '90 Days', '180 Days']

fig = go.Figure()

for period, label in zip(periods, period_labels):
    returns = summary_df[period].dropna() * 100
    fig.add_trace(go.Box(y=returns, name=label, boxmean=True))

fig.add_hline(y=0, line_dash='dash', line_color='red')

fig.update_layout(
    title='Return Distribution by Holding Period',
    yaxis_title='Return (%)',
    height=500
)
fig.show()

print("\nRETURNS BY HOLDING PERIOD")
print("="*70)
print(f"{'Period':<12} {'Median':>10} {'Mean':>10} {'Win Rate':>10} {'Worst':>10} {'Best':>10}")
print("-"*70)
for period, label in zip(periods, period_labels):
    returns = summary_df[period].dropna()
    print(f"{label:<12} {returns.median()*100:>9.1f}% {returns.mean()*100:>9.1f}% "
          f"{(returns > 0).mean()*100:>9.0f}% {returns.min()*100:>9.1f}% {returns.max()*100:>9.1f}%")

---
## 4. Individual Entry Analysis - Zoom In

In [ ]:
def plot_single_entry(entry_idx, lookback=30, lookahead=90):
    """Plot a single entry with context."""
    entry = entry_analysis[entry_idx]
    entry_date = entry['entry_date']
    entry_price = entry['entry_price']
    
    # Get price window
    entry_iloc = close.index.get_loc(entry_date)
    start_iloc = max(0, entry_iloc - lookback)
    end_iloc = min(len(close) - 1, entry_iloc + lookahead)
    
    window = close.iloc[start_iloc:end_iloc + 1]
    
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.7, 0.3],
                        subplot_titles=[f'Entry {entry_idx + 1}: {entry_date.strftime("%Y-%m-%d")}', 'SOPR'])
    
    # Price
    fig.add_trace(go.Scatter(x=window.index, y=window.values, name='Price',
                             line=dict(color='blue', width=2)), row=1, col=1)
    
    # Entry point
    fig.add_trace(go.Scatter(
        x=[entry_date], y=[entry_price],
        mode='markers+text',
        marker=dict(symbol='star', size=20, color='green'),
        text=[f'ENTRY<br>${entry_price:,.0f}'],
        textposition='bottom center',
        name='Entry'
    ), row=1, col=1)
    
    # Max drawdown point
    dd_idx = entry['returns_path'].idxmin()
    dd_price = close.loc[dd_idx]
    fig.add_trace(go.Scatter(
        x=[dd_idx], y=[dd_price],
        mode='markers+text',
        marker=dict(symbol='triangle-down', size=15, color='red'),
        text=[f'MAX DD<br>{entry["max_drawdown"]*100:.1f}%'],
        textposition='bottom center',
        name='Max DD'
    ), row=1, col=1)
    
    # Max gain point
    gain_idx = entry['returns_path'].idxmax()
    gain_price = close.loc[gain_idx]
    fig.add_trace(go.Scatter(
        x=[gain_idx], y=[gain_price],
        mode='markers+text',
        marker=dict(symbol='triangle-up', size=15, color='green'),
        text=[f'MAX GAIN<br>{entry["max_gain"]*100:.1f}%'],
        textposition='top center',
        name='Max Gain'
    ), row=1, col=1)
    
    # Entry line
    fig.add_hline(y=entry_price, line_dash='dot', line_color='gray', row=1, col=1)
    
    # SOPR
    sopr_window = df.loc[window.index, 'sopr']
    fig.add_trace(go.Scatter(x=sopr_window.index, y=sopr_window.values, 
                             name='SOPR', line=dict(color='purple')), row=2, col=1)
    fig.add_hline(y=1, line_dash='dash', line_color='red', row=2, col=1)
    
    fig.update_layout(height=600, showlegend=True,
                      title_text=f"Entry Analysis: Max DD {entry['max_drawdown']*100:.1f}%, "
                                 f"Max Gain {entry['max_gain']*100:.1f}%")
    return fig

In [ ]:
# Show best entries (highest max gain)
print("TOP 5 BEST ENTRIES (by max gain):")
best_entries = summary_df.nlargest(5, 'max_gain')
print(best_entries[['entry_date', 'entry_price', 'max_drawdown', 'max_gain', 'days_to_max_gain']].to_string())

# Plot the best one
best_idx = summary_df['max_gain'].idxmax()
fig = plot_single_entry(best_idx)
fig.show()

In [ ]:
# Show worst entries (biggest drawdown)
print("\nTOP 5 WORST ENTRIES (by max drawdown):")
worst_entries = summary_df.nsmallest(5, 'max_drawdown')
print(worst_entries[['entry_date', 'entry_price', 'max_drawdown', 'max_gain', 'days_to_max_dd']].to_string())

# Plot the worst one
worst_idx = summary_df['max_drawdown'].idxmin()
fig = plot_single_entry(worst_idx)
fig.show()

In [ ]:
# Show a few more entries for context
print("\nSHOWING SAMPLE ENTRIES:")
sample_indices = [0, len(entry_analysis)//4, len(entry_analysis)//2, 3*len(entry_analysis)//4, -1]

for idx in sample_indices:
    if idx < len(entry_analysis):
        fig = plot_single_entry(idx)
        fig.show()

---
## 5. Entry Quality Classification

In [ ]:
# Classify entries
def classify_entry(row):
    """Classify entry quality based on outcomes."""
    if row['max_gain'] > 0.50 and row['max_drawdown'] > -0.15:
        return 'Excellent'  # Big gain, small drawdown
    elif row['max_gain'] > 0.30 and row['max_drawdown'] > -0.20:
        return 'Good'  # Decent gain, manageable drawdown
    elif row['max_gain'] > 0.15 and row['max_drawdown'] > -0.15:
        return 'OK'  # Small gain, small drawdown
    elif row['max_drawdown'] < -0.30:
        return 'Terrible'  # Huge drawdown
    elif row['max_gain'] < row['max_drawdown'] * -1:
        return 'Bad'  # Drawdown bigger than max gain
    else:
        return 'Mediocre'

summary_df['quality'] = summary_df.apply(classify_entry, axis=1)

print("ENTRY QUALITY DISTRIBUTION")
print("="*50)
quality_counts = summary_df['quality'].value_counts()
for quality, count in quality_counts.items():
    pct = count / len(summary_df) * 100
    print(f"{quality}: {count} entries ({pct:.0f}%)")

In [ ]:
# Scatter plot: Max Drawdown vs Max Gain
fig = go.Figure()

colors = {'Excellent': 'darkgreen', 'Good': 'green', 'OK': 'lightgreen',
          'Mediocre': 'gray', 'Bad': 'orange', 'Terrible': 'red'}

for quality in colors:
    subset = summary_df[summary_df['quality'] == quality]
    if len(subset) > 0:
        fig.add_trace(go.Scatter(
            x=subset['max_drawdown'] * 100,
            y=subset['max_gain'] * 100,
            mode='markers',
            marker=dict(size=10, color=colors[quality]),
            name=f"{quality} ({len(subset)})",
            hovertemplate="%{text}<br>Max DD: %{x:.1f}%<br>Max Gain: %{y:.1f}%<extra></extra>",
            text=[d.strftime('%Y-%m-%d') for d in subset['entry_date']]
        ))

# Add diagonal line (gain = -drawdown)
fig.add_trace(go.Scatter(
    x=[-60, 0], y=[60, 0],
    mode='lines',
    line=dict(color='gray', dash='dash'),
    name='Break-even line'
))

fig.update_layout(
    title='Entry Quality: Max Drawdown vs Max Gain<br><sup>Above the line = More gain than pain</sup>',
    xaxis_title='Max Drawdown (%)',
    yaxis_title='Max Gain (%)',
    height=600
)
fig.show()

---
## 6. Optimal Exit Strategy Insights

In [ ]:
# Based on the data, what would be optimal exit?
print("\nOPTIMAL EXIT STRATEGY INSIGHTS")
print("="*60)

# Stop loss recommendation
median_dd = summary_df['max_drawdown'].median()
pct_75_dd = summary_df['max_drawdown'].quantile(0.25)  # 75% of entries have DD better than this
print(f"\n1. STOP LOSS:")
print(f"   Median max drawdown: {median_dd*100:.1f}%")
print(f"   75% of entries have max DD better than: {pct_75_dd*100:.1f}%")
print(f"   → Recommended stop loss: {abs(pct_75_dd)*100:.0f}% to {abs(median_dd)*100:.0f}%")

# Profit target
median_gain = summary_df['max_gain'].median()
pct_25_gain = summary_df['max_gain'].quantile(0.25)  # 75% of entries have gain better than this
print(f"\n2. PROFIT TARGET (if using fixed target):")
print(f"   Median max gain: {median_gain*100:.1f}%")
print(f"   75% of entries reach at least: {pct_25_gain*100:.1f}%")
print(f"   → Conservative target: {pct_25_gain*100:.0f}%")
print(f"   → Aggressive target: {median_gain*100:.0f}%")

# Optimal hold period
print(f"\n3. HOLDING PERIOD:")
print(f"   Median days to max drawdown: {summary_df['days_to_max_dd'].median():.0f}")
print(f"   Median days to max gain: {summary_df['days_to_max_gain'].median():.0f}")

# Best returns by period
periods = ['return_7d', 'return_14d', 'return_30d', 'return_60d', 'return_90d']
period_medians = {p: summary_df[p].median() for p in periods}
best_period = max(period_medians, key=period_medians.get)
print(f"   Best median return: {best_period.replace('return_', '').replace('d', ' days')} ({period_medians[best_period]*100:.1f}%)")

In [ ]:
# Risk/Reward analysis
print("\n4. RISK/REWARD RATIOS:")
print(f"   {'Scenario':<30} {'Risk':<10} {'Reward':<10} {'Ratio':<10}")
print("-"*65)

# If we use tight stop
tight_stop = 0.08
tight_reward = summary_df[summary_df['max_drawdown'] > -tight_stop]['max_gain'].median()
print(f"   {'Tight stop (8%)':<30} {tight_stop*100:<10.0f}% {tight_reward*100:<10.1f}% {tight_reward/tight_stop:<10.1f}")

# If we use medium stop
med_stop = 0.12
med_reward = summary_df[summary_df['max_drawdown'] > -med_stop]['max_gain'].median()
print(f"   {'Medium stop (12%)':<30} {med_stop*100:<10.0f}% {med_reward*100:<10.1f}% {med_reward/med_stop:<10.1f}")

# If we use wide stop
wide_stop = 0.20
wide_reward = summary_df[summary_df['max_drawdown'] > -wide_stop]['max_gain'].median()
print(f"   {'Wide stop (20%)':<30} {wide_stop*100:<10.0f}% {wide_reward*100:<10.1f}% {wide_reward/wide_stop:<10.1f}")

In [ ]:
# Win rates if we exited at different points
print("\n5. WIN RATE BY EXIT STRATEGY:")
print(f"   {'Exit Strategy':<40} {'Win Rate':<15}")
print("-"*60)

for period, label in [('return_7d', 'Hold 7 days'),
                       ('return_14d', 'Hold 14 days'),
                       ('return_30d', 'Hold 30 days'),
                       ('return_60d', 'Hold 60 days'),
                       ('return_90d', 'Hold 90 days')]:
    returns = summary_df[period].dropna()
    win_rate = (returns > 0).mean()
    print(f"   {label:<40} {win_rate*100:<15.0f}%")

# Max gain capture
print(f"   {'Perfect exit (at max gain)':<40} {'100%':<15}")

---
## 7. When Do Bad Entries Happen?

In [ ]:
# Look at terrible entries - what do they have in common?
terrible = summary_df[summary_df['quality'] == 'Terrible']
good_plus = summary_df[summary_df['quality'].isin(['Excellent', 'Good'])]

print("\nTERRIBLE ENTRIES ANALYSIS")
print("="*60)
print(f"\nTerrible entries: {len(terrible)}")
print("\nDates:")
for _, row in terrible.iterrows():
    print(f"  {row['entry_date'].strftime('%Y-%m-%d')} - DD: {row['max_drawdown']*100:.1f}%, Gain: {row['max_gain']*100:.1f}%")

In [ ]:
# Plot terrible entries on the price chart
fig = go.Figure()

fig.add_trace(go.Scatter(x=close.index, y=close, name='BTC Price',
                         line=dict(color='lightblue', width=1)))

# All entries as small dots
all_entry_dates = summary_df['entry_date']
all_entry_prices = [close.loc[d] for d in all_entry_dates]
fig.add_trace(go.Scatter(
    x=all_entry_dates, y=all_entry_prices,
    mode='markers',
    marker=dict(size=6, color='gray', opacity=0.5),
    name='All Entries'
))

# Terrible entries as big red X
if len(terrible) > 0:
    terrible_prices = [close.loc[d] for d in terrible['entry_date']]
    fig.add_trace(go.Scatter(
        x=terrible['entry_date'], y=terrible_prices,
        mode='markers+text',
        marker=dict(symbol='x', size=15, color='red', line=dict(width=2)),
        text=[f"{row['max_drawdown']*100:.0f}%" for _, row in terrible.iterrows()],
        textposition='bottom center',
        name='Terrible Entries'
    ))

# Excellent entries as big green stars
excellent = summary_df[summary_df['quality'] == 'Excellent']
if len(excellent) > 0:
    excellent_prices = [close.loc[d] for d in excellent['entry_date']]
    fig.add_trace(go.Scatter(
        x=excellent['entry_date'], y=excellent_prices,
        mode='markers+text',
        marker=dict(symbol='star', size=15, color='green'),
        text=[f"+{row['max_gain']*100:.0f}%" for _, row in excellent.iterrows()],
        textposition='top center',
        name='Excellent Entries'
    ))

fig.update_layout(
    title='Entry Quality on Price Chart<br><sup>Red X = Terrible, Green Star = Excellent</sup>',
    yaxis_type='log',
    height=600
)
fig.show()

---
## 8. Summary & Recommendations

In [ ]:
print("\n" + "="*70)
print("ENTRY ANALYSIS SUMMARY")
print("="*70)

print(f"\n📊 ENTRIES ANALYZED: {len(summary_df)}")

print(f"\n📈 ENTRY QUALITY:")
for quality in ['Excellent', 'Good', 'OK', 'Mediocre', 'Bad', 'Terrible']:
    count = len(summary_df[summary_df['quality'] == quality])
    pct = count / len(summary_df) * 100
    print(f"   {quality}: {count} ({pct:.0f}%)")

good_rate = len(summary_df[summary_df['quality'].isin(['Excellent', 'Good', 'OK'])]) / len(summary_df)
print(f"\n   → {good_rate*100:.0f}% of entries are OK or better")

print(f"\n📉 DRAWDOWN STATS:")
print(f"   Median: {summary_df['max_drawdown'].median()*100:.1f}%")
print(f"   Mean: {summary_df['max_drawdown'].mean()*100:.1f}%")
print(f"   Worst: {summary_df['max_drawdown'].min()*100:.1f}%")

print(f"\n📈 GAIN STATS:")
print(f"   Median: {summary_df['max_gain'].median()*100:.1f}%")
print(f"   Mean: {summary_df['max_gain'].mean()*100:.1f}%")
print(f"   Best: {summary_df['max_gain'].max()*100:.1f}%")

print(f"\n🎯 RECOMMENDATIONS:")
print(f"   1. Stop Loss: {abs(summary_df['max_drawdown'].quantile(0.25))*100:.0f}% (covers 75% of entries)")
print(f"   2. Trailing Stop: Let winners run beyond {summary_df['max_gain'].quantile(0.25)*100:.0f}%")
print(f"   3. Hold Period: Max gain typically reached in {summary_df['days_to_max_gain'].median():.0f} days")

print("\n" + "="*70)

In [ ]:
# Save analysis
summary_df.to_csv('../data/sopr_entry_analysis.csv', index=False)
print("Saved entry analysis to ../data/sopr_entry_analysis.csv")